Notebook 06b — LSTM & Transformer (stabilised training)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import glob
import json
import time
import math

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", device)
if device.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("\n*** WARNING: no GPU detected. Runtime -> Change runtime type -> GPU, then re-run. ***")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/AI_Trading_Dissertation"
FINAL_PROJECT_PATH = PROJECT_PATH + "/Final_Project"

SPLITS_PATH = FINAL_PROJECT_PATH + "/02_Data/Splits"
MODELS_PATH = FINAL_PROJECT_PATH + "/03_Models"
RESULTS_PATH = FINAL_PROJECT_PATH + "/04_Results"
PREDICTIONS_PATH = RESULTS_PATH + "/Predictions"
DL_PROGRESS_PATH = RESULTS_PATH + "/DL_Progress_v2"

for sub in ["LSTM_v2", "Transformer_v2"]:
    os.makedirs(os.path.join(MODELS_PATH, sub), exist_ok=True)
os.makedirs(DL_PROGRESS_PATH, exist_ok=True)

LABELS = ["Buy", "Hold", "Sell"]
LABEL_TO_INT = {l: i for i, l in enumerate(LABELS)}

# configuration
WINDOW = 64             # bars of history fed to the model
BATCH_SIZE = 256
MAX_EPOCHS = 60
PATIENCE = 10           # early stopping patience (was 5 - too aggressive in run 1)
LEARNING_RATE = 3e-4    # was 1e-3 - too large, caused divergence on ONGC/M&M
VAL_FRACTION = 0.10     # last 10% of TRAIN, chronological, for early stopping only
SEEDS = [42, 1337]      # two initialisations, best validation loss kept
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 1e-3


print(f"Window {WINDOW} bars | batch {BATCH_SIZE} | max {MAX_EPOCHS} epochs | patience {PATIENCE}")
print(f"LR {LEARNING_RATE} with {WARMUP_EPOCHS}-epoch warmup | weight decay {WEIGHT_DECAY} | seeds {SEEDS}")
print(f"Random-guess reference loss: ln(3) = {np.log(3):.4f}")


Window 64 bars | batch 256 | max 60 epochs | patience 10
LR 0.0003 with 3-epoch warmup | weight decay 0.001 | seeds [42, 1337]
Random-guess reference loss: ln(3) = 1.0986


Data loading and windowing


In [ ]:
train_files = sorted(glob.glob(os.path.join(SPLITS_PATH, "*_train.parquet")))
stock_names = [os.path.basename(f).replace("_train.parquet", "") for f in train_files]

_sample = pd.read_parquet(train_files[0])
EXCLUDE = {"datetime", "open", "high", "low", "close", "volume", "tick_count",
           "future_return", "target_threshold", "target_signal"}
FEATURE_COLUMNS = [c for c in _sample.columns if c not in EXCLUDE]
N_FEATURES = len(FEATURE_COLUMNS)

print(f"{len(stock_names)} stocks, {N_FEATURES} features")


10 stocks, 22 features


In [ ]:
class SequenceDataset(Dataset):
    """
    Windows of shape (WINDOW, n_features). The label for a window ending at row i
    is the target at row i - the model sees bars [i-WINDOW+1 .. i] and predicts
    the target that was computed at bar i (which looks 4 bars forward).
    """
    def __init__(self, X, y, window):
        self.X = np.ascontiguousarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.window = window

    def __len__(self):
        return len(self.X) - self.window + 1

    def __getitem__(self, idx):
        seq = self.X[idx: idx + self.window]          # (window, n_features)
        label = self.y[idx + self.window - 1]         # label at the LAST bar of the window
        return torch.from_numpy(seq), label


def prepare_stock(stock):
    train_df = pd.read_parquet(os.path.join(SPLITS_PATH, f"{stock}_train.parquet"))
    test_df = pd.read_parquet(os.path.join(SPLITS_PATH, f"{stock}_test.parquet"))

    # chronological inner validation split, used ONLY for early stopping
    cut = int(len(train_df) * (1 - VAL_FRACTION))
    inner_train = train_df.iloc[:cut]
    val = train_df.iloc[cut:]

    # Scaler fitted on inner-train ONLY - never on val or test
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(inner_train[FEATURE_COLUMNS].values)
    X_va = scaler.transform(val[FEATURE_COLUMNS].values)
    X_te = scaler.transform(test_df[FEATURE_COLUMNS].values)

    y_tr = inner_train["target_signal"].map(LABEL_TO_INT).values
    y_va = val["target_signal"].map(LABEL_TO_INT).values
    y_te = test_df["target_signal"].map(LABEL_TO_INT).values

    ds_tr = SequenceDataset(X_tr, y_tr, WINDOW)
    ds_va = SequenceDataset(X_va, y_va, WINDOW)
    ds_te = SequenceDataset(X_te, y_te, WINDOW)

    # class weights from the training labels only
    counts = np.bincount(y_tr, minlength=len(LABELS)).astype(float)
    weights = counts.sum() / (len(LABELS) * np.maximum(counts, 1))
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

    return ds_tr, ds_va, ds_te, class_weights, test_df, scaler


## STEP 2 — The two architectures

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden=64, layers=2, n_classes=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden, num_layers=layers,
            batch_first=True, dropout=dropout if layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):                 # x: (batch, window, features)
        out, _ = self.lstm(x)
        last = out[:, -1, :]              # final timestep's hidden state
        return self.head(last)


class PositionalEncoding(nn.Module):
    # Sinusoidal position signal - a Transformer has no inherent notion of order so position must be injected explicitly.
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]


class TransformerClassifier(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, layers=2, n_classes=3, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        h = self.input_proj(x)
        h = self.pos(h)
        h = self.encoder(h)
        pooled = h.mean(dim=1)            # average across the window
        return self.head(pooled)


Training loop with early stopping

In [ ]:
def train_once(model, ds_tr, ds_va, class_weights, tag, seed, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = model.to(device)

    loader_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    loader_va = DataLoader(ds_va, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    # Linear warmup then cosine decay. Warmup is the specific fix for run 1's
    # divergence: it stops the optimiser taking large destructive steps in the
    # first epochs, which is exactly what ONGC's rising validation loss showed.
    steps_per_epoch = max(len(loader_tr), 1)

    def lr_lambda(step):
        epoch = step / steps_per_epoch
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1e-8) / WARMUP_EPOCHS
        progress = (epoch - WARMUP_EPOCHS) / max(MAX_EPOCHS - WARMUP_EPOCHS, 1)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)

    best_val = float("inf")
    best_state = None
    stale = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total, n = 0.0, 0
        for xb, yb in loader_tr:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimiser.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
            scheduler.step()
            total += loss.item() * len(yb); n += len(yb)
        train_loss = total / max(n, 1)

        model.eval()
        total, n, correct = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in loader_va:
                xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                logits = model(xb)
                total += criterion(logits, yb).item() * len(yb)
                correct += (logits.argmax(1) == yb).sum().item()
                n += len(yb)
        val_loss = total / max(n, 1)
        val_acc = correct / max(n, 1)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_acc": val_acc})

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            stale = 0
            mark = "  *best"
        else:
            stale += 1
            mark = ""

        if verbose and (epoch <= 5 or epoch % 5 == 0 or mark):
            print(f"    {tag} s{seed} ep {epoch:2d}  train {train_loss:.4f}  val {val_loss:.4f}  acc {val_acc:.4f}{mark}")

        if stale >= PATIENCE:
            if verbose:
                print(f"    early stop at epoch {epoch} (best was epoch {best_epoch}, val {best_val:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val, best_epoch, history


def train_model(builder, ds_tr, ds_va, class_weights, tag, verbose=True):
    """Train once per seed, keep whichever run reached the lowest validation loss.
    Guards against a single unlucky initialisation - which is what several of
    run 1's failures look like."""
    best = None
    for seed in SEEDS:
        model, val_loss, best_epoch, history = train_once(builder(), ds_tr, ds_va, class_weights, tag, seed, verbose)
        converged = best_epoch > 2      # best epoch 1-2 means it never really trained
        print(f"    -> seed {seed}: best val {val_loss:.4f} at epoch {best_epoch}"
              f"{'' if converged else '   <-- did NOT converge'}")
        if best is None or val_loss < best[1]:
            best = (model, val_loss, best_epoch, history)
    print(f"    kept: val loss {best[1]:.4f} (best epoch {best[2]})")
    return best[0], best[3], best[1], best[2]


@torch.no_grad()
def predict(model, ds):
    model.eval()
    loader = DataLoader(ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)
    all_probs, all_true = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        all_probs.append(torch.softmax(model(xb), dim=1).cpu().numpy())
        all_true.append(yb.numpy())
    return np.concatenate(all_probs), np.concatenate(all_true)


STEP 4 — Evaluation, matching Notebook 04 exactly.

In [ ]:
def evaluate(y_true_int, y_pred_int, stock, model_name, train_seconds):
    y_true = np.array([LABELS[i] for i in y_true_int])
    y_pred = np.array([LABELS[i] for i in y_pred_int])

    traded = np.isin(y_pred, ["Buy", "Sell"])
    n_trades = int(traded.sum())
    traded_acc = accuracy_score(y_true[traded], y_pred[traded]) if n_trades else np.nan

    row = {
        "Stock": stock, "Model": model_name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision_Macro": round(precision_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0), 4),
        "Recall_Macro": round(recall_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0), 4),
        "F1_Macro": round(f1_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0), 4),
        "F1_Weighted": round(f1_score(y_true, y_pred, average="weighted", labels=LABELS, zero_division=0), 4),
        "Traded_Accuracy": round(traded_acc, 4) if not np.isnan(traded_acc) else np.nan,
        "N_Trade_Signals": n_trades,
        "Trade_Signal_Rate": round(n_trades / len(y_pred), 4),
        "Train_Seconds": round(train_seconds, 1),
    }
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    return row, cm


In [ ]:
def progress_file(stock):
    return os.path.join(DL_PROGRESS_PATH, f"{stock}_dl_v2_done.json")


all_dl_rows = []
all_dl_confusion = {}
overall_start = time.time()

for i, stock in enumerate(stock_names, 1):
    print(f"\n{'=' * 72}")
    print(f"[{i}/{len(stock_names)}]  {stock}")
    print("=" * 72)

    if os.path.exists(progress_file(stock)):
        print("  already completed in an earlier run - skipping")
        with open(progress_file(stock)) as f:
            saved = json.load(f)
        all_dl_rows.extend(saved["rows"])
        continue

    ds_tr, ds_va, ds_te, class_weights, test_df, scaler = prepare_stock(stock)
    print(f"  sequences: train {len(ds_tr):,}  val {len(ds_va):,}  test {len(ds_te):,}")
    print(f"  class weights: {class_weights.cpu().numpy().round(3)}")

    stock_rows = []

    # Predictions frame - aligned to the bars that actually have a full window behind them
    preds_out = test_df.iloc[WINDOW - 1:][["datetime", "close", "future_return", "target_signal"]].copy().reset_index(drop=True)

    for model_name, builder in [
        ("LSTM", lambda: LSTMClassifier(N_FEATURES)),
        ("Transformer", lambda: TransformerClassifier(N_FEATURES)),
    ]:
        print(f"\n  --- {model_name} ---")
        t0 = time.time()
        model, history, best_val, best_epoch = train_model(builder, ds_tr, ds_va, class_weights, model_name)
        secs = time.time() - t0

        probs, y_true_int = predict(model, ds_te)
        y_pred_int = probs.argmax(axis=1)

        row, cm = evaluate(y_true_int, y_pred_int, stock, model_name, secs)
        row["Best_Val_Loss"] = round(best_val, 4)
        row["Best_Epoch"] = int(best_epoch)
        row["Converged"] = bool(best_epoch > 2)
        stock_rows.append(row)
        all_dl_confusion[(stock, model_name)] = cm

        preds_out[f"pred_{model_name}"] = [LABELS[i] for i in y_pred_int]
        for k, lab in enumerate(LABELS):
            preds_out[f"prob_{model_name}_{lab}"] = probs[:, k]

        torch.save(model.state_dict(), os.path.join(MODELS_PATH, model_name + "_v2", f"{stock}_{model_name.lower()}_v2.pt"))

        print(f"  {model_name:12s} acc={row['Accuracy']:.4f}  F1={row['F1_Macro']:.4f}  "
              f"val_loss={best_val:.4f}  best_ep={best_epoch}  ({secs/60:.1f} min)")

    preds_out.to_parquet(os.path.join(PREDICTIONS_PATH, f"{stock}_dl_v2_predictions.parquet"), index=False)

    with open(progress_file(stock), "w") as f:
        json.dump({"rows": stock_rows}, f)

    all_dl_rows.extend(stock_rows)
    elapsed = (time.time() - overall_start) / 60
    print(f"\n  saved. elapsed {elapsed:.1f} min, ~{elapsed / i * (len(stock_names) - i):.0f} min remaining")

print(f"\n\nALL DONE in {(time.time() - overall_start) / 60:.1f} minutes.")



[1/10]  Eicher_Motors
  already completed in an earlier run - skipping

[2/10]  HDFCB
  already completed in an earlier run - skipping

[3/10]  Hindustan_Unilever
  already completed in an earlier run - skipping

[4/10]  ICICIBC
  already completed in an earlier run - skipping

[5/10]  INFOSYS
  already completed in an earlier run - skipping

[6/10]  ITC
  already completed in an earlier run - skipping

[7/10]  Mahindra_and_Mahindra
  already completed in an earlier run - skipping

[8/10]  ONGC
  already completed in an earlier run - skipping

[9/10]  Reliance
  already completed in an earlier run - skipping

[10/10]  TCS
  already completed in an earlier run - skipping


ALL DONE in 0.0 minutes.


Results

In [ ]:
dl_df = pd.DataFrame(all_dl_rows)
dl_df.to_csv(os.path.join(RESULTS_PATH, "DL_Metrics_AllStocks_v2.csv"), index=False)

dl_summary = (
    dl_df.groupby("Model")[["Accuracy", "Precision_Macro", "Recall_Macro", "F1_Macro", "Traded_Accuracy", "Train_Seconds"]]
    .mean().round(4)
)
print("Deep learning models, averaged across 10 stocks:\n")
display(dl_summary)


Deep learning models, averaged across 10 stocks:



,Accuracy,Precision_Macro,Recall_Macro,F1_Macro,Traded_Accuracy,Train_Seconds
Model,,,,,,
LSTM,0.3664,0.3725,0.3833,0.3254,0.4037,80.29
Transformer,0.3606,0.3744,0.3811,0.3092,0.4063,138.55


# Are sophisticated models actually better?

In [ ]:
tree_df = pd.read_csv(os.path.join(RESULTS_PATH, "Model_Metrics_AllStocks.csv"))
combined = pd.concat([tree_df, dl_df], ignore_index=True)
combined.to_csv(os.path.join(RESULTS_PATH, "All_Models_Metrics.csv"), index=False)

full_summary = (
    combined.groupby("Model")[["Accuracy", "Precision_Macro", "Recall_Macro", "F1_Macro", "Traded_Accuracy"]]
    .mean().round(4).sort_values("F1_Macro", ascending=False)
)
full_summary.to_csv(os.path.join(RESULTS_PATH, "Final_Tables", "Table1_Classification_AllModels.csv"))

print("ALL MODELS, ranked by macro F1:\n")
display(full_summary)

print("\nPer-stock F1:\n")
pivot = combined.pivot_table(index="Stock", columns="Model", values="F1_Macro")
order = [c for c in ["Baseline_Random", "Baseline_Majority", "Baseline_Momentum",
                     "RandomForest", "XGBoost", "LSTM", "Transformer"] if c in pivot.columns]
pivot = pivot[order]
pivot.to_csv(os.path.join(RESULTS_PATH, "Final_Tables", "F1_AllModels_ByStock.csv"))
display(pivot.round(4))

print("\nWins per model (best F1 on each stock):")
display(pivot.idxmax(axis=1).value_counts())


ALL MODELS, ranked by macro F1:



,Accuracy,Precision_Macro,Recall_Macro,F1_Macro,Traded_Accuracy
Model,,,,,
RandomForest,0.3938,0.3956,0.3983,0.3841,0.3985
XGBoost,0.3892,0.3978,0.3786,0.3732,0.3816
Baseline_Random,0.3390,0.3346,0.3346,0.3345,0.3619
LSTM,0.3664,0.3725,0.3833,0.3254,0.4037
Transformer,0.3606,0.3744,0.3811,0.3092,0.4063
Baseline_Momentum,0.3391,0.2841,0.3157,0.2815,0.3496
Baseline_Majority,0.3604,0.1201,0.3333,0.1766,0.3604



Per-stock F1:



Model,Baseline_Random,Baseline_Majority,Baseline_Momentum,RandomForest,XGBoost,LSTM,Transformer
Stock,,,,,,,
Eicher_Motors,0.3357,0.1757,0.2750,0.3911,0.3695,0.3022,0.2672
HDFCB,0.3379,0.1739,0.2794,0.3650,0.3620,0.3328,0.3241
Hindustan_Unilever,0.3350,0.1770,0.2802,0.3739,0.3798,0.3667,0.3623
ICICIBC,0.3316,0.1776,0.2847,0.3683,0.3613,0.2521,0.2020
INFOSYS,0.3397,0.1700,0.2755,0.3849,0.4034,0.3855,0.3906
ITC,0.3387,0.1812,0.2906,0.3977,0.3764,0.3844,0.3823
Mahindra_and_Mahindra,0.3394,0.1750,0.2801,0.3740,0.3723,0.1592,0.1576
ONGC,0.3299,0.1810,0.2861,0.3913,0.3707,0.2974,0.2371
Reliance,0.3288,0.1837,0.2862,0.3998,0.3611,0.3919,0.3843



Wins per model (best F1 on each stock):


,count
RandomForest,8
XGBoost,2


In [ ]:
# Training cost comparison - axis of evaluation
cost_compare = combined.groupby("Model")["Train_Seconds"].mean().sort_values()
print("Mean training time per stock (seconds):\n")
display(cost_compare.round(1))

best_tree = full_summary.loc[[m for m in full_summary.index if m in ("RandomForest", "XGBoost")]].iloc[0]
best_dl = full_summary.loc[[m for m in full_summary.index if m in ("LSTM", "Transformer")]].iloc[0]
best_tree_name = [m for m in full_summary.index if m in ("RandomForest", "XGBoost")][0]
best_dl_name = [m for m in full_summary.index if m in ("LSTM", "Transformer")][0]

print(f"\nBest tree model : {best_tree_name:14s} F1 = {best_tree['F1_Macro']:.4f}")
print(f"Best DL model   : {best_dl_name:14s} F1 = {best_dl['F1_Macro']:.4f}")
diff = best_dl["F1_Macro"] - best_tree["F1_Macro"]
print(f"Difference      : {diff:+.4f}  ->  {'deep learning wins' if diff > 0 else 'tree ensembles win'}")


Mean training time per stock (seconds):



,Train_Seconds
Model,
Baseline_Momentum,0.0
Baseline_Random,0.0
Baseline_Majority,0.0
XGBoost,12.4
RandomForest,60.2
LSTM,80.3
Transformer,138.6



Best tree model : RandomForest   F1 = 0.3841
Best DL model   : LSTM           F1 = 0.3254
Difference      : -0.0587  ->  tree ensembles win


Confusion matrices

In [ ]:
for model_name in ["LSTM", "Transformer"]:
    print(f"\n{'#' * 60}\n# {model_name}\n{'#' * 60}")
    for stock in stock_names:
        if (stock, model_name) not in all_dl_confusion:
            continue
        cm = all_dl_confusion[(stock, model_name)]
        print(f"\n{stock}:")
        print(pd.DataFrame(cm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS]))

cm_rows = []
for (stock, model_name), cm in all_dl_confusion.items():
    for a, tl in enumerate(LABELS):
        for b, pl in enumerate(LABELS):
            cm_rows.append({"Stock": stock, "Model": model_name, "True": tl, "Predicted": pl, "Count": int(cm[a, b])})
pd.DataFrame(cm_rows).to_csv(os.path.join(RESULTS_PATH, "Confusion_Matrices_DL_v2.csv"), index=False)
print("\nSaved.")



############################################################
# LSTM
############################################################

############################################################
# Transformer
############################################################

Saved.


In [ ]:
v1 = pd.read_csv(os.path.join(RESULTS_PATH, "DL_Metrics_AllStocks.csv"))
v2 = dl_df.copy()

comp = (
    v1.groupby("Model")["F1_Macro"].mean().rename("Run1_F1")
    .to_frame()
    .join(v2.groupby("Model")["F1_Macro"].mean().rename("Run2_F1"))
)
comp["Change"] = (comp["Run2_F1"] - comp["Run1_F1"]).round(4)
comp = comp.round(4)

print("Mean macro F1 across 10 stocks:\n")
display(comp)

print("\nPer-stock, run 1 -> run 2:\n")
per = (
    v1.pivot_table(index="Stock", columns="Model", values="F1_Macro")
      .join(v2.pivot_table(index="Stock", columns="Model", values="F1_Macro"), lsuffix="_run1", rsuffix="_run2")
)
cols = [c for c in ["LSTM_run1", "LSTM_run2", "Transformer_run1", "Transformer_run2"] if c in per.columns]
display(per[cols].round(4))


Mean macro F1 across 10 stocks:



,Run1_F1,Run2_F1,Change
Model,,,
LSTM,0.3162,0.3254,0.0092
Transformer,0.3388,0.3092,-0.0297



Per-stock, run 1 -> run 2:



Model,LSTM_run1,LSTM_run2,Transformer_run1,Transformer_run2
Stock,,,,
Eicher_Motors,0.2916,0.3022,0.3177,0.2672
HDFCB,0.3193,0.3328,0.3498,0.3241
Hindustan_Unilever,0.3876,0.3667,0.3654,0.3623
ICICIBC,0.1929,0.2521,0.3573,0.2020
INFOSYS,0.3765,0.3855,0.3690,0.3906
ITC,0.3912,0.3844,0.3668,0.3823
Mahindra_and_Mahindra,0.1616,0.1592,0.2746,0.1576
ONGC,0.2651,0.2974,0.2452,0.2371
Reliance,0.3849,0.3919,0.3622,0.3843


In [ ]:
# Convergence audit - the evidence that run 1's failures were optimisation, not signal
if "Converged" in v2.columns:
    print("Run 2 convergence check (best epoch > 2 means the model genuinely trained):\n")
    audit = v2[["Stock", "Model", "Best_Epoch", "Best_Val_Loss", "Converged", "F1_Macro"]].copy()
    display(audit.sort_values(["Model", "Stock"]).round(4))

    n_bad = int((~v2["Converged"]).sum())
    print(f"\nModels that still failed to converge in run 2: {n_bad} of {len(v2)}")
    print(f"Random-guess reference loss ln(3) = {np.log(3):.4f} - compare against Best_Val_Loss above.")

    # Which stocks improved most - should be the four that failed in run 1
    merged = v1[["Stock", "Model", "F1_Macro"]].merge(
        v2[["Stock", "Model", "F1_Macro"]], on=["Stock", "Model"], suffixes=("_v1", "_v2")
    )
    merged["Improvement"] = (merged["F1_Macro_v2"] - merged["F1_Macro_v1"]).round(4)
    print("\nLargest improvements (run 1 failures should dominate this list):\n")
    display(merged.sort_values("Improvement", ascending=False).head(8))


Run 2 convergence check (best epoch > 2 means the model genuinely trained):



,Stock,Model,Best_Epoch,Best_Val_Loss,Converged,F1_Macro
0,Eicher_Motors,LSTM,7,1.0765,True,0.3022
2,HDFCB,LSTM,9,1.0575,True,0.3328
4,Hindustan_Unilever,LSTM,8,1.0657,True,0.3667
6,ICICIBC,LSTM,5,1.0815,True,0.2521
8,INFOSYS,LSTM,23,1.0589,True,0.3855
10,ITC,LSTM,9,1.0762,True,0.3844
12,Mahindra_and_Mahindra,LSTM,14,1.0782,True,0.1592
14,ONGC,LSTM,2,1.0848,False,0.2974
16,Reliance,LSTM,20,1.0696,True,0.3919
18,TCS,LSTM,22,1.0575,True,0.3816



Models that still failed to converge in run 2: 1 of 20
Random-guess reference loss ln(3) = 1.0986 - compare against Best_Val_Loss above.

Largest improvements (run 1 failures should dominate this list):



,Stock,Model,F1_Macro_v1,F1_Macro_v2,Improvement
6,ICICIBC,LSTM,0.1929,0.2521,0.0592
14,ONGC,LSTM,0.2651,0.2974,0.0323
17,Reliance,Transformer,0.3622,0.3843,0.0221
9,INFOSYS,Transformer,0.3690,0.3906,0.0216
11,ITC,Transformer,0.3668,0.3823,0.0155
2,HDFCB,LSTM,0.3193,0.3328,0.0135
0,Eicher_Motors,LSTM,0.2916,0.3022,0.0106
8,INFOSYS,LSTM,0.3765,0.3855,0.0090


all-model ranking (stabilised deep learning)

In [ ]:
tree = pd.read_csv(os.path.join(RESULTS_PATH, "Model_Metrics_AllStocks.csv"))
final_all = pd.concat([tree, v2], ignore_index=True)
final_all.to_csv(os.path.join(RESULTS_PATH, "All_Models_Metrics_v2.csv"), index=False)

ranking = (
    final_all.groupby("Model")[["Accuracy", "Precision_Macro", "Recall_Macro", "F1_Macro", "Traded_Accuracy"]]
    .mean().round(4).sort_values("F1_Macro", ascending=False)
)
os.makedirs(os.path.join(RESULTS_PATH, "Final_Tables"), exist_ok=True)
ranking.to_csv(os.path.join(RESULTS_PATH, "Final_Tables", "Table1_Classification_Final.csv"))

print("FINAL RANKING — all models, stabilised deep learning:\n")
display(ranking)

pivot = final_all.pivot_table(index="Stock", columns="Model", values="F1_Macro")
order = [c for c in ["Baseline_Random", "Baseline_Majority", "Baseline_Momentum",
                     "RandomForest", "XGBoost", "LSTM", "Transformer"] if c in pivot.columns]
pivot = pivot[order]
pivot.to_csv(os.path.join(RESULTS_PATH, "Final_Tables", "F1_AllModels_ByStock_Final.csv"))
print("\nPer-stock F1:\n")
display(pivot.round(4))

print("\nWins per model:")
display(pivot.idxmax(axis=1).value_counts())


FINAL RANKING — all models, stabilised deep learning:



,Accuracy,Precision_Macro,Recall_Macro,F1_Macro,Traded_Accuracy
Model,,,,,
RandomForest,0.3938,0.3956,0.3983,0.3841,0.3985
XGBoost,0.3892,0.3978,0.3786,0.3732,0.3816
Baseline_Random,0.3390,0.3346,0.3346,0.3345,0.3619
LSTM,0.3664,0.3725,0.3833,0.3254,0.4037
Transformer,0.3606,0.3744,0.3811,0.3092,0.4063
Baseline_Momentum,0.3391,0.2841,0.3157,0.2815,0.3496
Baseline_Majority,0.3604,0.1201,0.3333,0.1766,0.3604



Per-stock F1:



Model,Baseline_Random,Baseline_Majority,Baseline_Momentum,RandomForest,XGBoost,LSTM,Transformer
Stock,,,,,,,
Eicher_Motors,0.3357,0.1757,0.2750,0.3911,0.3695,0.3022,0.2672
HDFCB,0.3379,0.1739,0.2794,0.3650,0.3620,0.3328,0.3241
Hindustan_Unilever,0.3350,0.1770,0.2802,0.3739,0.3798,0.3667,0.3623
ICICIBC,0.3316,0.1776,0.2847,0.3683,0.3613,0.2521,0.2020
INFOSYS,0.3397,0.1700,0.2755,0.3849,0.4034,0.3855,0.3906
ITC,0.3387,0.1812,0.2906,0.3977,0.3764,0.3844,0.3823
Mahindra_and_Mahindra,0.3394,0.1750,0.2801,0.3740,0.3723,0.1592,0.1576
ONGC,0.3299,0.1810,0.2861,0.3913,0.3707,0.2974,0.2371
Reliance,0.3288,0.1837,0.2862,0.3998,0.3611,0.3919,0.3843



Wins per model:


,count
RandomForest,8
XGBoost,2
